# Debug Viewer: Colab Proxy & Tailscale Check

外部サービス（Cloudflare等）を使用せず、Colab標準のプロキシ機能とTailscaleの状態を確認します。

**確認項目:**
1.  **Colab Proxy**: `google.colab.output.eval_js` によるURL解決（タイムアウト保護付き）。
2.  **Tailscale**: Tailscaleが接続されているか、およびそのURL。
3.  **Localhost**: ローカル環境（VS Code等）で実行している場合のURL。

In [6]:
import os
import sys
import subprocess
import time
import signal
from IPython.display import display, Markdown

# === Environment Detection ===
try:
    from google.colab import output
    from google.colab.output import eval_js
    IS_COLAB = True
    print("Environment: Google Colab detected.")
except ImportError:
    IS_COLAB = False
    print("Environment: Local / VS Code detected.")

PORT = 8000
VIEWER_DIR = '/content/viewer' if IS_COLAB else './viewer'

# === 1. Setup & Start Server ===
def setup_server():
    print(f"Setting up viewer in {VIEWER_DIR}...")
    if not os.path.exists(VIEWER_DIR):
        subprocess.run(["git", "clone", "https://github.com/antimatter15/splat", VIEWER_DIR], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    with open(f"{VIEWER_DIR}/simple_server.py", 'w') as f:
        f.write('''
from http.server import HTTPServer, SimpleHTTPRequestHandler
import sys
class CORSRequestHandler(SimpleHTTPRequestHandler):
    def end_headers(self):
        self.send_header('Cross-Origin-Opener-Policy', 'same-origin')
        self.send_header('Cross-Origin-Embedder-Policy', 'require-corp')
        self.send_header('Access-Control-Allow-Origin', '*')
        super().end_headers()
if __name__ == '__main__':
    HTTPServer(('', 8000), CORSRequestHandler).serve_forever()
''')

    # Kill existing
    subprocess.run(f"fuser -k {PORT}/tcp", shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    
    # Start
    print(f"Starting HTTP Server on port {PORT}...")
    subprocess.Popen([sys.executable, "simple_server.py", str(PORT)], cwd=VIEWER_DIR, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(2)

# === 2. Method: Colab Proxy (with Timeout) ===
def get_colab_proxy():
    print("Resolving Proxy URL for port 8000...")
    if not IS_COLAB: return None
    
    def handler(signum, frame):
        raise TimeoutError("Proxy resolution timed out")
    
    signal.signal(signal.SIGALRM, handler)
    signal.alarm(5)
    
    try:
        url = eval_js(f"google.colab.kernel.proxyPort({PORT})")
        if url and not url.endswith('/'): url += '/'
        return url
    except TimeoutError:
        print("-> Colab Proxy resolution TIMED OUT.")
        return None
    except Exception as e:
        print(f"-> Colab Proxy failed: {e}")
        return None
    finally:
        signal.alarm(0)

# === 3. Method: Tailscale ===
def check_tailscale():
    # Simple check if tailscaled is running
    res = subprocess.run("ps aux | grep tailscaled | grep -v grep", shell=True, stdout=subprocess.PIPE)
    if res.returncode == 0:
        return f"http://colab:{PORT}/index.html"
    return None

# === MAIN EXECUTION ===
setup_server()

# Check Proxy
proxy_url = get_colab_proxy()

# Check Tailscale
tailscale_url = check_tailscale()

print("\n" + "="*30)
print("VIEWER ACCESS LINKS")
print("="*30)

if proxy_url:
    full_url = f"{proxy_url}index.html"
    display(Markdown(f"## [OPEN 3DGS VIEWER (Colab Proxy)]({full_url})"))
else:
    print("\u274c Colab Proxy Not Available (Timed out or Failed)")

if tailscale_url:
    display(Markdown(f"## [OPEN VIA TAILSCALE]({tailscale_url})"))
else:
    print("\u26a0 Tailscale Not Connected")

if not IS_COLAB:
     display(Markdown(f"## [OPEN LOCALHOST](http://localhost:{PORT}/index.html)"))


Environment: Google Colab detected.
Setting up viewer in /content/viewer...
Starting HTTP Server on port 8000...
Resolving Proxy URL for port 8000...
-> Colab Proxy resolution TIMED OUT.

VIEWER ACCESS LINKS
❌ Colab Proxy Not Available (Timed out or Failed)
⚠ Tailscale Not Connected
